In [1]:
#run this first
!pip install -q condacolab
import condacolab
condacolab.install()
#restart after install is normal


📢 Announcement 📢
condacolab==0.2 will be released soon! Try it with:

    !pip install -q https://github.com/conda-incubator/condacolab/archive/main.zip
    import condacolab
    condacolab.install()

0.2.x introduces a new installation method based on Pixi, with customizable Python versions.
This may be breaking for your workflow. If that's the case, please report it at
https://github.com/conda-incubator/condacolab and pin your `pip install` command to
condacolab==0.1 as a workaround.

⏬ Downloading https://github.com/conda-forge/miniforge/releases/download/26.3.2-3/Miniforge3-26.3.2-3-Linux-x86_64.sh...
📦 Installing...
📌 Adjusting configuration...
🩹 Patching environment...
⏲ Done in 0:00:06
🔁 Restarting kernel...


### env setup

In [1]:
!git clone https://github.com/alvrian/BRIO.git
!cd BRIO

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

Cloning into 'BRIO'...
remote: Enumerating objects: 184, done.
remote: Counting objects: 100% (95/95), done.
remote: Compressing objects: 100% (40/40), done.
remote: Total 184 (delta 76), reused 62 (delta 55), pack-reused 89 (from 1)
Receiving objects: 100% (184/184), 7.64 MiB | 19.22 MiB/s, done.
Resolving deltas: 100% (98/98), done.
Mounted at /content/drive


In [ ]:
#ganti dataset
%cd BRIO

! cp /content/drive/MyDrive/Dataset/cnndm_subset_20k.zip .
! unzip -q cnndm_subset_20k.zip
! mv cnndm_subset_20k cnndm

/content/BRIO


In [ ]:
#checking unzipped file
import os
import json
from glob import glob

print(len(os.listdir('/content/BRIO/cnndm/diverse/train')))
print(len(os.listdir('/content/BRIO/cnndm/diverse/test')))
print(len(os.listdir('/content/BRIO/cnndm/diverse/val')))

os.makedirs('./cache', exist_ok=True)
os.makedirs('./result', exist_ok=True)

print(f'Working directory: {os.getcwd()}')
print(f'Contents: {os.listdir(".")}')

#checking format
expected_keys = {
    "article", "abstract", "candidates",
    "article_untok", "abstract_untok", "candidates_untok"
}

for split in ['train', 'val', 'test']:
    path = f'./cnndm/diverse/{split}'
    if os.path.exists(path):
        files = glob(os.path.join(path, "*.json"))
        invalid_count = 0
        for file_path in files:
            try:
                with open(file_path, 'r') as f:
                    data = json.load(f)
                if not expected_keys.issubset(data.keys()):
                    invalid_count += 1
            except Exception:
                invalid_count += 1
        print(f'{split}: checked {len(files)} files, {invalid_count} files failed structure validation.')
    else:
        print(f'{split}: NOT FOUND at {path}')

In [ ]:
! pwd
! conda create --name env --file spec-file.txt
! conda run -n env pip install -r requirements.txt
!conda run -n env python -c "import nltk;nltk.download('punkt')"

#using new requirements.txt file
# ! conda run -n env pip install "protobuf==3.20.3"
# ! conda run -n env pip install --upgrade transformers requests
# ! conda run -n env pip install --upgrade torch

In [ ]:
#sanity check for conda enviroment
!conda run -n env python -c "import torch; print('CUDA Available:', torch.cuda.is_available()); print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')"
!conda run -n env python -c "import torch, transformers, tensorboard; print('Imports successful!')"


In [ ]:
! git clone https://github.com/neulab/compare-mt.git
! cd compare-mt && conda run -n env pip install -r requirements.txt
! cd compare-mt && conda run -n env python setup.py install

### exp

In [ ]:
#run after experiment rerun

# !rm -rf ~/.cache/huggingface/transformers/

In [ ]:
#training
!conda run --no-capture-output -n env python main.py \
    --cuda --gpuid 0 --config cnndm -l

start...
start processing data in ./cnndm/diverse/train
start processing data in ./cnndm/diverse/val
done extracting data for cnndm for BRIO
start building model
BartScorer has generative capabilities, as `prepare_inputs_for_generation` is explicitly overwritten. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you are not the owner of the model architecture class, please contact the model code owner to update it.
Namespa

In [ ]:
# !conda run --no-capture-output -n env python main.py \
#     --cuda --gpuid 0 --config cnndm -e --model_pt cnndm/model_generation.bin -g

start...




start processing data in ./cnndm/diverse/test


BartScorer has generative capabilities, as `prepare_inputs_for_generation` is explicitly overwritten. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you are not the owner of the model architecture class, please contact the model code owner to update it.

main.py:86: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value)

In [ ]:
import os
import shutil
from datetime import datetime

# Relative path based on CWD (/content/BRIO)
source_cache = './cache'
drive_dest_dir = '/content/drive/MyDrive/BRIO_checkpoint'

#please change based on experiment
dataset_name = 'cnndm'

timestamp = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
archive_base_name = f'cache_{timestamp}_{dataset_name}'
temp_archive_path = f'./{archive_base_name}'

if os.path.exists(source_cache):
    print("Compressing cache directory...")
    created_zip = shutil.make_archive(temp_archive_path, 'zip', source_cache)
    print(f"Archive created locally at: {created_zip}")

    try:
        if not os.path.exists('/content/drive/MyDrive'):
            from google.colab import drive
            drive.mount('/content/drive', force_remount=False)

        os.makedirs(drive_dest_dir, exist_ok=True)
        destination_zip = os.path.join(drive_dest_dir, f"{archive_base_name}.zip")

        shutil.move(created_zip, destination_zip)
        print(f"Cache successfully saved to Google Drive at: {destination_zip}")

    except Exception as e:
        # Fallback: Only triggers local download if Drive upload fails
        print(f"\n[Error] Google Drive upload failed: {e}")
        print("Initiating local browser download as fallback...")
        from google.colab import files
        files.download(created_zip)

else:
    print(f"Error: Source cache directory not found at {os.path.abspath(source_cache)}")

In [ ]:
import os
import shutil
from datetime import datetime

# Relative path based on CWD (/content/BRIO)
source_cache = './result'
drive_dest_dir = '/content/drive/MyDrive/BRIO_checkpoint'

#please change based on experiment
dataset_name = 'cnndm'

timestamp = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
archive_base_name = f'result_{timestamp}_{dataset_name}'
temp_archive_path = f'./{archive_base_name}'

if os.path.exists(source_cache):
    print("Compressing cache directory...")
    created_zip = shutil.make_archive(temp_archive_path, 'zip', source_cache)
    print(f"Archive created locally at: {created_zip}")

    try:
        if not os.path.exists('/content/drive/MyDrive'):
            from google.colab import drive
            drive.mount('/content/drive', force_remount=False)

        os.makedirs(drive_dest_dir, exist_ok=True)
        destination_zip = os.path.join(drive_dest_dir, f"{archive_base_name}.zip")

        shutil.move(created_zip, destination_zip)
        print(f"result successfully saved to Google Drive at: {destination_zip}")

    except Exception as e:
        # Fallback: Only triggers local download if Drive upload fails
        print(f"\n[Error] Google Drive upload failed: {e}")
        print("Initiating local browser download as fallback...")
        from google.colab import files
        files.download(created_zip)

else:
    print(f"Error: Source cache directory not found at {os.path.abspath(source_cache)}")

In [ ]:
# ! conda run -n env pip freeze > updated_requirements.txt